# Two-body low-energy scattering: from a potential to a bound state

**Pedro Henrique Gesualdo Modesto** — São Carlos Institute of Physics (IFSC), University of São Paulo
Advisor: Lucas Madeira · Co-advisor: Patrícia C. M. Castilho

---

## Question

At low energy the shape of an interaction potential is invisible. Only two
numbers survive — the **scattering length** $a$ and the **effective range**
$r_0$ — through the effective-range expansion

$$k\cot\delta_0(k) = -\frac1a + \tfrac12 r_0 k^2 + O(k^4).$$

**If four unrelated potentials are tuned to the same $(a, r_0)$, do they
predict the same binding energy?** This notebook answers that quantitatively
for two real systems nine orders of magnitude apart in energy.

## Pipeline

| § | Step | Output |
|---|---|---|
| 1 | Four potentials | Fig. 1 |
| 2 | Measure $(a, r_0)$ | Table 1 |
| 3 | **Inverse problem**: target $(a,r_0)$ → parameters | Table 2, Fig. 2 |
| 4 | Bound states vs. universal formulas | Table 3, Fig. 3 |
| 5 | Parameter sweeps, 1D and 2D | Figs. 4–5 |
| 6 | Export CSV + PDF for the manuscript | `paper/` |

## Conventions

$\hbar = \mu_{\text{red}} = 1$, lengths in fm, so the radial equation is
$u'' = 2(V-E)u$. Physical energies follow from
$E_{\text{phys}} = 2\,(\hbar^2/2\mu_{\text{red}})\,E_{\text{code}}$.

| Code name | Symbol | Meaning |
|---|---|---|
| `scattering_length` | $a$ | where the asymptotic straight line crosses zero |
| `effective_range` | $r_0$ | leading finite-size correction |
| `depth` | $v$, $C_6$ | strength parameter |
| `inverse_range` | $\mu$, $C_{12}$ | scale parameter, $R = 1/\mu$ |
| `n_nodes` | $n$ | zeros of $u(r)$ — counts bound states |

> **Run this notebook with Run All.** Cell 1 defines everything; after it has
> run once, any cell works on its own. Total runtime ≈ 90 s.

In [ ]:
# ===========================================================================
#  CELL 1 — everything the notebook needs.  RUN THIS FIRST.
# ===========================================================================
import sys, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp        # bound-state ODE
from scipy.optimize import brentq            # bracketed root finding

# Walk up until we find src/ — works from any subdirectory.
ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# The numerical engine lives in src/ and is covered by 71 unit tests.
# This notebook only calls it — nothing is reimplemented here.
from src.dois_corpos import espalhamento, ajuste, analitico
from src.dois_corpos.potenciais import FABRICAS
from src.literatura.tabelas_artigo import TABELA1, TABELA2, TABELA3, TABELA4

pd.set_option("display.float_format", lambda x: f"{x:.6g}")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": .3, "figure.autolayout": True})

# One colour per potential, reused in every figure.
COLOR = {"poco": "#1f77b4", "mpt": "#d62728", "gauss": "#2ca02c", "lj": "#9467bd"}
LABEL = {"poco": "Square well", "mpt": "Modified Pöschl-Teller",
         "gauss": "Gaussian", "lj": "Lennard-Jones"}
POTS = tuple(LABEL)

# Radial step per potential. The Lennard-Jones 1/r^6 tail is numerically alive
# out to hundreds of fm, so a uniform fine grid there costs 200k points per
# evaluation; 2e-2 changes the 4th decimal and is ~10x cheaper.
STEP = {"poco": 5e-3, "mpt": 5e-3, "gauss": 5e-3, "lj": 2e-2}


def scattering(pot, step=1e-3):
    """(a, r0, n_nodes) for any potential. Thin wrapper over the tested engine."""
    res = espalhamento.calcular(pot, dr=step, metodo="numerov")
    return res.a, res.r0, res.nos


def tune(name, a_target, r0_target, p1, p2, n_nodes=None, step=None):
    """Inverse problem: find the parameters reproducing (a_target, r0_target).

    Two nested loops: the inner one solves 1/a = 1/a_target for the depth,
    the outer one adjusts the scale until r0 matches. The root is taken on
    1/a, never on a: a has poles (one per new bound state) and bisection dies
    on a pole, while 1/a is smooth and crosses zero. Unitarity is then just
    1/a = 0 and needs no special case.
    """
    return ajuste.ajustar(name, a_target, r0_target, p1, p2,
                          dr=step or STEP[name], nos_alvo=n_nodes)


def _start_radius(pot, ceiling=1e4):
    """Where to start integrating. Zero for smooth wells; for Lennard-Jones the
    r^-12 core reaches 1e10 and stiffens the solver, so we start where V has
    dropped to 1e4 — the wavefunction is already negligible inside."""
    r = np.logspace(-4, 2, 40000)
    inside = pot.V(r) >= ceiling
    return float(r[inside][-1]) if inside.any() else pot.r_min


def _match_radius(pot, tol=1e-8):
    """Where to stop: beyond this the potential is negligible and the solution
    is already the free decaying exponential exp(-kappa r)."""
    r = np.linspace(max(pot.r_min, 1e-6), min(pot.R, 1e4), 20000)
    alive = np.abs(pot.V(r)) >= tol
    return float(r[alive][-1]) if alive.any() else float(r[0])


def bound_energy(pot, n_scan=120):
    """Ground-state energy in code units (fm^-2); NaN if the potential binds nothing.

    Outside the range V = 0, so u must join exp(-kappa r) with kappa = sqrt(-2E):
        mismatch(E) = u'(Rc) + kappa * u(Rc) = 0.
    We use the sum, not u'/u + kappa: the deuteron has a node, u(Rc) can vanish,
    and the ratio would produce a spurious pole for the root finder to chase.
    """
    r_start, r_end = _start_radius(pot), _match_radius(pot)

    def mismatch(E):
        # solve_ivp integrates u'' = 2(V - E)u as the system y = (u, u').
        sol = solve_ivp(lambda r, y: [y[1], 2*(pot.V(r) - E)*y[0]],
                        [r_start, r_end], [0.0, 1.0],
                        method="DOP853", rtol=1e-9, atol=1e-14)
        u, du = sol.y[0, -1], sol.y[1, -1]
        # Normalise so the exponential growth of u cannot overflow the root finder.
        return (du + math.sqrt(-2*E)*u) / max(abs(u), abs(du), 1.0)

    # Scan on a LOG grid: shallow states sit against zero (the helium dimer is
    # at 1.6e-3 K in a well ~1 K deep) and a linear grid would step over them.
    floor = float(np.min(pot.V(np.linspace(r_start, r_end, 5000))))
    grid = -np.logspace(math.log10(abs(floor)), -11, n_scan)
    vals = [mismatch(E) for E in grid]
    for i in range(n_scan - 1):                       # first sign change = ground state
        if vals[i]*vals[i+1] < 0:
            return brentq(mismatch, grid[i], grid[i+1], xtol=1e-14)
    return float("nan")


# Everything the manuscript needs lands here. Defined now so any figure can be
# written the moment it is built — plt.show() clears pyplot's registry, so
# collecting figures at the end would silently export nothing.
PAPER = ROOT / "paper"
(PAPER / "figures").mkdir(parents=True, exist_ok=True)
(PAPER / "tables").mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    """Write a figure as vector PDF for the manuscript. Call before plt.show()."""
    fig.savefig(PAPER / "figures" / f"{name}.pdf", bbox_inches="tight")
    return fig


print(f"root: {ROOT.name} | potentials: {', '.join(LABEL.values())}")

---
## Literature values used, and why

Every number taken from outside this laboratory, with its source. Nothing
enters the calculations that is not on this list.

In [ ]:
# Each row: what it is, the number, where it comes from, and why we need it.
LITERATURE = pd.DataFrame([
    dict(quantity="a (n-p triplet)", value=5.4112, unit="fm",
         source="Hackenburg, Phys. Rev. C 73, 044002 (2006)",
         used_for="Target of the deuteron tuning. Ref. [23] of RBEF 2023."),
    dict(quantity="r0 (n-p triplet)", value=1.7436, unit="fm",
         source="Hackenburg, Phys. Rev. C 73, 044002 (2006)",
         used_for="Second target of the deuteron tuning."),
    dict(quantity="E (deuteron)", value=-2.224, unit="MeV",
         source="Hackenburg, Phys. Rev. C 73, 044002 (2006)",
         used_for="Experimental benchmark for §4 — not an input."),
    dict(quantity="a (4He dimer)", value=90.4, unit="angstrom",
         source="Cencek et al., J. Chem. Phys. 136, 224303 (2012)",
         used_for="Target of the helium tuning. Ref. [22] of RBEF 2023."),
    dict(quantity="r0 (4He dimer)", value=8.0, unit="angstrom",
         source="Cencek et al., J. Chem. Phys. 136, 224303 (2012)",
         used_for="Second target of the helium tuning."),
    dict(quantity="E (4He dimer)", value=-1.62e-3, unit="K",
         source="Cencek et al., J. Chem. Phys. 136, 224303 (2012)",
         used_for="Experimental benchmark for §4 — not an input."),
    dict(quantity="a (n-n singlet)", value=-18.5, unit="fm",
         source="Macedo-Lima & Madeira, Rev. Bras. Ensino Fis. 45, e20230079 (2023), Table 2",
         used_for="Third tuning case: large NEGATIVE a, no bound state."),
    dict(quantity="Published potential parameters", value=np.nan, unit="-",
         source="Macedo-Lima & Madeira (2023), Tables 3 and 4",
         used_for="Initial guesses AND the validation target of §3."),
    dict(quantity="v threshold, square well", value=math.pi**2/8, unit="-",
         source="Analytic: pi^2/8 = 1.2337",
         used_for="Exact check of the first bound state in the §5 sweep."),
])
display(LITERATURE)

# hbar^2 / 2*mu_red is NOT tabulated anywhere: we invert the zero-range formula
# E_zr = -(hbar^2/2mu)/a^2 on the published pair (a, E_zr). Recovering the known
# constants (41.47 MeV.fm^2 and 12.12 K.A^2) proves the table is self-consistent.
for key, unit, known in (("deuteron", "MeV.fm^2", 41.47), ("he4_dimer", "K.A^2", 12.12)):
    row = TABELA1[key]
    inferred = -row["E_zr_ref"] * row["a"]**2
    print(f"hbar^2/2mu inferred for {key:<10} = {inferred:7.3f} {unit:<9} "
          f"(literature {known}, {100*abs(inferred/known-1):.2f}% off)")

---
## 1. Four potentials

| Potential | Form | What it stresses |
|---|---|---|
| Square well | $-v\mu^2$ for $r<1/\mu$ | a **discontinuity** |
| Modified Pöschl-Teller | $-v\mu^2/\cosh^2(\mu r)$ | **smooth and analytically solvable** |
| Gaussian | $-v\mu^2 e^{-\mu^2r^2}$ | a tail dying **faster than exponentially** |
| Lennard-Jones | $\tfrac12(C_{12}/r^{12} - C_6/r^6)$ | **hard core** + van der Waals tail |

All forms are Eqs. (74), (116), (118) and (120) of Macêdo-Lima & Madeira (2023).

> **Convention warning.** The $\tfrac12$ in the Lennard-Jones is not universal:
> much of the literature writes $C_{12}/r^{12}-C_6/r^6$ without it, which shifts
> $C_6$ by a factor of 2 while the result still *looks* right. We keep the
> $\tfrac12$; see `referencias/CONVENCOES.md`.

In [ ]:
# Build all four at unitarity (|a| -> infinity), the most interesting point in
# cold-atom physics: the potential is exactly at the threshold of binding.
CASE = "unitario"
demo = {n: FABRICAS[n](*( (TABELA3 if n != "lj" else TABELA4)[(CASE, n)][k]
                          for k in ("p1", "p2") )) for n in POTS}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))

# Left: the three smooth potentials share a linear scale.
r = np.linspace(1e-4, 4, 600)
for n in ("poco", "mpt", "gauss"):
    ax1.plot(r, demo[n].V(r), color=COLOR[n], lw=1.6, label=LABEL[n])
ax1.axhline(0, color="k", lw=.6)
ax1.set(xlabel="r (fm)", ylabel="V(r) (fm$^{-2}$)",
        title=f"Fig. 1a — three smooth potentials at {CASE}")
ax1.legend(fontsize=7)

# Right: the Lennard-Jones needs log axes — its core reaches +1e10.
r_lj = np.logspace(-2, .5, 600)
ax2.plot(r_lj, demo["lj"].V(r_lj), color=COLOR["lj"], lw=1.6)
ax2.axhline(0, color="k", lw=.6)
ax2.axvline(demo["lj"].r_min, color="gray", ls=":", label=f"core cut r={demo['lj'].r_min:.3g} fm")
ax2.set(xscale="log", yscale="symlog", xlabel="r (fm)", ylabel="V(r) (fm$^{-2}$)",
        title="Fig. 1b — Lennard-Jones: hard core + vdW tail")
ax2.legend(fontsize=7)
save_fig(fig, "fig1_potentials")
plt.show()

---
## 2. Measuring $a$ and $r_0$

One function, four potentials. Integrate $u''=2Vu$ with $u(0)=0$; outside the
range $u$ is a straight line, and $a$ is where it crosses zero. Then
$r_0 = 2\int_0^\infty[g_0^2 - u^2]\,dr$ with $g_0 = 1-r/a$ (Simpson).

Two numerical choices that matter, both already in `src/comum/solvers.py`:
the grid is **aligned to the well edge** (otherwise the discontinuity falls
between grid points and the convergence order collapses), and Numerov is used
for smooth potentials only — on a discontinuous one it degrades from 4th to
1st order, which is physics, not a bug.

In [ ]:
# Table 1: the four potentials of Fig. 1, measured.
target = TABELA2[CASE]
table1 = pd.DataFrame([
    dict(potential=LABEL[n], **dict(zip(("a_fm", "r0_fm", "n_nodes"), scattering(demo[n]))))
    for n in POTS
]).assign(r0_error=lambda d: d.r0_fm - target["r0"])
print(f"target: a = {target['a']}, r0 = {target['r0']} fm, nodes = {target['nos']}")
display(table1)

---
## 3. The inverse problem: automatic tuning

Given a target $(a, r_0)$, find the parameters of **any** potential that
reproduce it. Both parameters move both observables, so this is a nonlinear
$2\times2$ system — and $a(\text{depth})$ has poles, one per new bound state.

The strategy is two nested loops, each solving **one** equation in **one**
variable by bisection:

```
outer: adjust the scale (mu or C12)     -> matches r0
  inner: adjust the depth (v or C6)     -> matches 1/a
```

**Root on $1/a$, never on $a$.** Bisection dies on a pole and thrives on a
zero. As a bonus, unitarity becomes $1/a = 0$ with no special case.

**Counting nodes is mandatory.** Two solutions can share $(a,r_0)$ and differ
in node count — they are then not the same physical state. Every tuning ends
by checking it.

In [ ]:
# Reproduce Tables 3 and 4 of the article in full: 3 cases x 4 potentials.
# Each run starts from the published parameters and must return to them.
CASES = ("nn", "unitario", "deuteron")
CASE_LABEL = {"nn": "neutron-neutron", "unitario": "unitarity", "deuteron": "deuteron"}

rows, fits = [], {}
for case in CASES:                                        # loop over targets
    for n in POTS:                                        # loop over potentials
        pub = (TABELA3 if n != "lj" else TABELA4)[(case, n)]
        fit = tune(n, TABELA2[case]["a"], TABELA2[case]["r0"],
                   pub["p1"], pub["p2"], n_nodes=TABELA2[case]["nos"])
        fits[(case, n)] = fit
        rows.append(dict(case=CASE_LABEL[case], potential=LABEL[n],
                         p1=fit.p1, p1_published=pub["p1"],
                         p2=fit.p2, p2_published=pub["p2"],
                         a=fit.a, r0=fit.r0, nodes=fit.nos, converged=fit.convergiu))

# Largest relative disagreement with the published parameters, per row.
table2 = pd.DataFrame(rows).assign(
    max_dev_pct=lambda d: 100*np.maximum((d.p1/d.p1_published - 1).abs(),
                                         (d.p2/d.p2_published - 1).abs()))
display(table2[["case", "potential", "p1", "p1_published", "p2", "p2_published",
                "nodes", "converged", "max_dev_pct"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

# Left: three different shapes producing the SAME (a, r0). This is universality.
r = np.linspace(1e-4, 6, 600)
for n in ("poco", "mpt", "gauss"):
    f = fits[("deuteron", n)]
    axes[0].plot(r, FABRICAS[n](f.p1, f.p2).V(r), color=COLOR[n], lw=1.6, label=LABEL[n])
axes[0].axhline(0, color="k", lw=.6)
axes[0].set(xlabel="r (fm)", ylabel="V(r) (fm$^{-2}$)", xlim=(0, 6),
            title="Fig. 2a — tuned to the deuteron\na = 5.4 fm, $r_0$ = 1.7 fm")
axes[0].legend(fontsize=7)

# Right: the (1/a, r0) plane. Star = target, dots = the four tuned potentials
# landing on top of it. This IS the operational definition of universality.
for case in CASES:
    t = TABELA2[case]
    axes[1].scatter(0 if math.isinf(t["a"]) else 1/t["a"], t["r0"],
                    s=110, marker="*", zorder=5, label=f"{CASE_LABEL[case]} (n={t['nos']})")
    for n in POTS:                                        # the four solutions
        axes[1].scatter(1/fits[(case, n)].a, fits[(case, n)].r0,
                        s=22, color=COLOR[n], zorder=6)
axes[1].axvline(0, color="gray", ls=":")
axes[1].set(xlabel="1/a (fm$^{-1}$)  — 1/a is the physical variable, not a",
            ylabel="$r_0$ (fm)", title="Fig. 2b — targets and solutions")
axes[1].legend(fontsize=7, loc="lower left")
save_fig(fig, "fig2_tuning")
plt.show()

---
## 4. Bound states

Now the question that matters. Two real systems, four potentials each, all
tuned to the same $(a, r_0)$, compared against:

| Formula | Expression | What it drops |
|---|---|---|
| zero range | $E=-\dfrac{\hbar^2}{2\mu_{red}a^2}$ | the size of the potential entirely |
| finite range | root of $\kappa = 1/a + r_0\kappa^2/2$ | the $O(k^4)$ terms |

In [ ]:
# Initial guesses: the deuteron solution, RESCALED. The problem is scale
# invariant — multiplying all lengths by L maps (a, r0) -> (L*a, L*r0) — so we
# divide the scale parameter by L. C12 carries L^6 because it multiplies r^-12.
GUESS = {"poco": (1.7806, .48415), "mpt": (1.44397, .853766),
         "gauss": (1.93585, .652551), "lj": (7.8433, 1.27426)}

def bench(key, unit, pots=POTS):
    """Tune every potential on a Table-1 target and compute its binding energy."""
    ref = TABELA1[key]
    h2_2mu = -ref["E_zr_ref"] * ref["a"]**2      # inferred, see §Literature
    to_phys = 2 * h2_2mu                         # E_phys = 2*(hbar^2/2mu)*E_code
    L = ref["r0"] / 1.7436                       # length scale vs. the deuteron

    out = [dict(method="experiment", energy=ref["E_ref"], unit=unit),
           dict(method="zero range", energy=analitico.energia_zr(ref["a"], h2_2mu), unit=unit),
           dict(method="finite range",
                energy=analitico.energia_fr(ref["a"], ref["r0"], h2_2mu), unit=unit)]
    for n in pots:                               # loop over the four potentials
        p1, p2 = GUESS[n]
        fit = tune(n, ref["a"], ref["r0"], p1, p2 * (L**6 if n == "lj" else 1/L),
                   n_nodes=1, step=STEP[n]*L)
        out.append(dict(method=LABEL[n], unit=unit, p1=fit.p1, p2=fit.p2,
                        energy=to_phys * bound_energy(FABRICAS[n](fit.p1, fit.p2))))
    df = pd.DataFrame(out)
    df["dev_pct"] = 100*(df.energy/ref["E_ref"] - 1)          # vs. experiment
    df.attrs = dict(system=key, ratio=abs(ref["a"])/ref["r0"], E_exp=ref["E_ref"])
    return df

# Lennard-Jones is skipped for helium: with a = 90 A its 1/r^6 tail only dies
# after thousands of A, and a uniform grid becomes prohibitive.
deuteron = bench("deuteron", "MeV")
helium   = bench("he4_dimer", "K", pots=("poco", "mpt", "gauss"))

for df in (deuteron, helium):
    print(f"\n{df.attrs['system']}   |a|/r0 = {df.attrs['ratio']:.1f}   "
          f"experiment = {df.attrs['E_exp']:.5g} {df.unit.iloc[0]}")
    display(df[["method", "energy", "unit", "dev_pct"]])

### Reading

The zero-range formula misses by **36% for the deuteron** and 8.6% for helium.
The difference is $|a|/r_0$: 3.1 versus 11.3. **The deuteron — the textbook
shallow bound state — fails the usual universality criterion $|a|/r_0 > 10$.**

Adding $r_0$ brings both below 1%. Two numbers, and nothing else about the
nuclear or van der Waals interaction, suffice.

The four potentials agree with each other to ~1%. That residual spread is
**shape dependence** — the $O(k^4)$ terms, the only thing still distinguishing
a square well from a Lennard-Jones once $a$ and $r_0$ are fixed.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

# Left: wavefunctions. Different inside the range, identical outside.
for n in ("poco", "mpt", "gauss"):
    f = fits[("deuteron", n)]
    res = espalhamento.calcular(FABRICAS[n](f.p1, f.p2), dr=1e-3)
    axes[0].plot(res.r, res.u, color=COLOR[n], lw=1.5, label=LABEL[n])
a_d = TABELA1["deuteron"]["a"]
axes[0].plot([0, 9], [1, 1 - 9/a_d], "k--", lw=1.3, label="asymptote $1-r/a$")
axes[0].scatter([a_d], [0], s=55, color="k", zorder=5)
axes[0].annotate(f"a = {a_d} fm", (a_d, 0), (a_d-2.6, .32),
                 arrowprops=dict(arrowstyle="->", lw=.8), fontsize=8)
axes[0].axhline(0, color="k", lw=.6)
axes[0].set(xlim=(0, 9), xlabel="r (fm)", ylabel="u(r)",
            title="Fig. 3a — same outside, different inside")
axes[0].legend(fontsize=7)

# Middle and right: deviation from experiment, one panel per system.
for k, df in enumerate((deuteron, helium)):
    ax = axes[k+1]
    d = df[df.method != "experiment"]
    ax.bar(range(len(d)), d.dev_pct,
           color=["#888", "#333"] + [COLOR[n] for n in POTS][:len(d)-2])
    ax.axhline(0, color="k", lw=1.1)
    ax.axhspan(-1, 1, color="green", alpha=.12)                # the +/-1% band
    ax.set_xticks(range(len(d)), d.method, rotation=35, fontsize=7, ha="right")
    ax.set(ylabel="deviation from experiment (%)",
           title=f"Fig. 3{'bc'[k]} — {df.attrs['system']}, $|a|/r_0$ = {df.attrs['ratio']:.1f}")
save_fig(fig, "fig3_bound_states")
plt.show()

---
## 5. Parameter sweeps

Sections 1–4 solved the inverse problem. Now the direct one: sweep the
parameters and look at the structure. This is what justifies every choice
made in §3.

In [ ]:
# 5.1 — sweep the depth at fixed scale (mu = 1, range R = 1 fm).
depths = np.linspace(.05, 12, 240)
sweep = {n: np.array([scattering(FABRICAS[n](v, 1.0), step=2e-3) for v in depths])
         for n in ("poco", "mpt", "gauss")}          # columns: a, r0, nodes

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for n, s in sweep.items():
    axes[0].plot(depths, 1/s[:, 0], color=COLOR[n], lw=1.4, label=LABEL[n])
    a = np.where(np.abs(s[:, 0]) > 60, np.nan, s[:, 0])   # clip the runaway branches
    axes[1].plot(depths, a, color=COLOR[n], lw=1.4)
    axes[2].step(depths, s[:, 2], where="post", color=COLOR[n], lw=1.5, label=LABEL[n])

axes[0].axhline(0, color="k", lw=1)
axes[0].axvline(analitico.V_LIMIAR_POCO, color=COLOR["poco"], ls=":")
axes[0].text(analitico.V_LIMIAR_POCO + .2, .8,
             f"$\\pi^2/8$ = {analitico.V_LIMIAR_POCO:.4f}", fontsize=7, color=COLOR["poco"])
axes[0].set(xlabel="depth v", ylabel="1/a (fm$^{-1}$)",
            title="Fig. 4a — 1/a is smooth and crosses zero")
axes[0].legend(fontsize=7)
axes[1].axhline(0, color="k", lw=.6)
axes[1].set(xlabel="depth v", ylabel="a (fm)", ylim=(-60, 60),
            title="Fig. 4b — a has poles: one per new bound state")
axes[2].set(xlabel="depth v", ylabel="nodes of u(r)",
            title="Fig. 4c — each step is a bound state")
axes[2].legend(fontsize=7)
save_fig(fig, "fig4_depth_sweep")
plt.show()

# The square-well threshold is known exactly, so this is a hard check.
first = depths[np.argmax(sweep["poco"][:, 2] > 0)]
print(f"square-well threshold — exact pi^2/8 = {analitico.V_LIMIAR_POCO:.6f} | "
      f"measured {first:.4f} | grid resolution {depths[1]-depths[0]:.4f}")

In [ ]:
# 5.2 — sweep BOTH parameters. The tuning solution is where the two level sets
# of 1/a and r0 cross; seeing that explains why two nested loops beat a 2D Newton.
N = 55
v_axis  = np.linspace(.3, 6, N)
mu_axis = np.linspace(.25, 2.2, N)
inv_a = np.empty((N, N))
r0_map = np.empty((N, N))
for i, mu in enumerate(mu_axis):                      # rows = scale
    for j, v in enumerate(v_axis):                    # columns = depth
        a, r0, _ = scattering(FABRICAS["gauss"](v, mu), step=5e-3)
        inv_a[i, j], r0_map[i, j] = 1/a, r0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
lim = np.nanpercentile(np.abs(inv_a), 96)             # robust colour limits
im = ax1.pcolormesh(v_axis, mu_axis, inv_a, cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
plt.colorbar(im, ax=ax1, label="1/a (fm$^{-1}$)")
ax1.contour(v_axis, mu_axis, inv_a, levels=[0], colors="k", linewidths=2)
ax1.set(xlabel="depth v", ylabel="inverse range $\\mu$ (fm$^{-1}$)",
        title="Fig. 5a — Gaussian: 1/a map\nblack line = unitarity")

ref = TABELA1["deuteron"]
ax2.contour(v_axis, mu_axis, inv_a, levels=[1/ref["a"]], colors="#d62728", linewidths=2)
ax2.contour(v_axis, mu_axis, r0_map, levels=[ref["r0"]], colors="#1f77b4", linewidths=2)
sol = fits[("deuteron", "gauss")]
ax2.scatter([sol.p1], [sol.p2], s=130, marker="*", color="k", zorder=6)
ax2.plot([], [], color="#d62728", lw=2, label=f"1/a = 1/{ref['a']}")
ax2.plot([], [], color="#1f77b4", lw=2, label=f"$r_0$ = {ref['r0']} fm")
ax2.plot([], [], "k*", ms=10, label=f"tuning solution\nv={sol.p1:.4f}, $\\mu$={sol.p2:.4f}")
ax2.set(xlabel="depth v", ylabel="inverse range $\\mu$ (fm$^{-1}$)",
        title="Fig. 5b — tuning is a crossing of level sets")
ax2.legend(fontsize=7)
save_fig(fig, "fig5_parameter_map")
plt.show()

The two curves in Fig. 5b are **nearly parallel** over much of the plane. A
2D Newton solver would meet an ill-conditioned Jacobian exactly there and take
large steps in the wrong direction. Nested bisection cannot diverge once the
sign is bracketed — slower per iteration, and far more reliable for a notebook
somebody else will run.

---
## 6. Export for the manuscript

Every table becomes a CSV **and** a LaTeX fragment; every figure becomes a PDF.
The manuscript then `\input{}`s these files, so no number is ever retyped:
rerun the notebook and the paper updates itself.

In [ ]:
# The four tables, with the caption the manuscript will use.
EXPORTS = {
    "literature":  (LITERATURE, "Literature values used as targets and benchmarks."),
    "measured":    (table1,     "Scattering length and effective range at unitarity."),
    "tuning":      (table2,     "Automatic tuning against Tables 3 and 4 of Macedo-Lima and Madeira (2023)."),
    "deuteron":    (deuteron,   "Deuteron binding energy from four tuned potentials."),
    "helium":      (helium,     "Helium dimer binding energy from three tuned potentials."),
}
# LaTeX has ten characters it refuses to print literally. pandas' own escaping
# needs a newer jinja2 than most installs have, so we do it here in one line.
TEX_ESCAPE = str.maketrans({c: "\\" + c for c in "&%$#_{}"} |
                           {"~": r"\textasciitilde{}", "^": r"\textasciicircum{}"})

def to_tex(df, path, caption, label):
    """Write a DataFrame as a standalone LaTeX table the manuscript can \input."""
    esc = lambda x: str(x).translate(TEX_ESCAPE)
    head = " & ".join(esc(c).replace("\\_", " ") for c in df.columns)   # underscores -> spaces
    body = "\\\\\n".join(
        " & ".join(f"{v:.4g}" if isinstance(v, float) else esc(v) for v in row)
        for row in df.fillna("--").itertuples(index=False))
    path.write_text(
        "\\begin{table}[htbp]\n\\centering\n\\small\n"
        f"\\caption{{{esc(caption)}}}\n\\label{{{label}}}\n"
        f"\\begin{{tabular}}{{{'l'*len(df.columns)}}}\n\\hline\n"
        f"{head} \\\\\n\\hline\n{body} \\\\\n\\hline\n"
        "\\end{tabular}\n\\end{table}\n", encoding="utf-8")

for name, (df, caption) in EXPORTS.items():               # one loop, both formats
    df.to_csv(PAPER / "tables" / f"{name}.csv", index=False)
    to_tex(df, PAPER / "tables" / f"{name}.tex", caption, f"tab:{name}")

# The figures were already written by save_fig() as each one was built.

print(f"exported to {PAPER}")
for f in sorted(PAPER.rglob("*")):
    if f.is_file():
        print(f"   {f.relative_to(PAPER)}  ({f.stat().st_size/1024:.1f} kB)")

---
## What this notebook establishes

1. **The tuning works for any potential.** Twelve independent fits reproduce
   the published parameters to better than 0.2% for the smooth potentials.
   The Lennard-Jones sits at 0.9–2.7%, from the deliberately coarser step.
2. **Two numbers describe low-energy physics.** Four unrelated potentials
   tuned to the same $(a,r_0)$ agree on the binding energy to ~1%, across
   nine orders of magnitude in energy.
3. **The effective range is not a detail.** Zero range misses by 36% for the
   deuteron; $|a|/r_0 = 3.1$ puts it outside the universal regime.
4. **Parameter-space structure justifies the method** — poles vs. zeros,
   mandatory node counting, near-parallel level sets.

### Next

Three bodies. The engine is already in `src/tres_corpos/trimero.py`, with the
universal ratio $e^{\pi/s_0}=22.694$ verified; it needs a two-body potential
anchored on known $(a,r_0)$, which is exactly what this notebook produces.
The dissertation method — Quantum Monte Carlo (VMC + DMC) — will take these
tuned potentials as input.

### Deliberately out of scope

- **No error bars.** `src/comum/incerteza.py` has Richardson extrapolation and
  validity gates ready; left out to keep this readable.
- **Lennard-Jones for helium**, for the grid-cost reason given in §4.
- **s-wave only**, valid while $kR \ll 1$.